# Video Demo
Here, we provide a demo on how to use our model to process a video and visualize the tracking results.

We selected an open hpop dance video from the internet to demonstrate our demo. You can also choose other custom videos. **Please note that it is crucial to select the appropriate trained MOTIP weights and configuration for different tracking scenarios.**

We process the video on NVIDIA RTX 3080Ti, achieving a nearly real-time tracking.

In [ ]:
import os
import sys
from pathlib import Path

MOTIP_ROOT = Path(
    "/root/autodl-tmp/project/basket_cut/MOTIP"
).resolve()

assert MOTIP_ROOT.is_dir(), f"MOTIP目录不存在: {MOTIP_ROOT}"
assert (MOTIP_ROOT / "utils").is_dir()
assert (MOTIP_ROOT / "models").is_dir()

os.chdir(MOTIP_ROOT)

motip_root_str = str(MOTIP_ROOT)
if motip_root_str not in sys.path:
    sys.path.insert(0, motip_root_str)

print("工作目录:", Path.cwd())
print("Python搜索路径首项:", sys.path[0])
from utils.nested_tensor import nested_tensor_from_tensor_list
from demo.colormap import get_color
from models.runtime_tracker import RuntimeTracker

print("MOTIP imports OK")

### System Environment
1. Modify the root path to the project path.
2. Make sure you have a cuda device available.

In [ ]:
import os
import sys
import torch


current_file_path = os.path.abspath("")
parent_dir = os.path.dirname(current_file_path)
sys.path.append(parent_dir)
os.chdir(parent_dir)
print(f"Current root path is set to {parent_dir}")

torch_version = torch.__version__
cuda_available = torch.cuda.is_available()

if not cuda_available:
    raise RuntimeError("CUDA is not available")

print(f"Hello! Welcome to use the video process demo. Your torch version is {torch_version} and CUDA is available.")

### Build our model

In [ ]:
import os
import sys
from pathlib import Path

import torch

MOTIP_ROOT = Path(
    "/root/autodl-tmp/project/basket_cut/MOTIP"
).resolve()

os.chdir(MOTIP_ROOT)

if str(MOTIP_ROOT) not in sys.path:
    sys.path.insert(0, str(MOTIP_ROOT))

from utils.misc import yaml_to_dict
from configs.util import load_super_config
from models.motip import build as build_model
from models.misc import load_checkpoint

config_path = (
    MOTIP_ROOT
    / "configs"
    / "r50_deformable_detr_motip_sportsmot.yaml"
)

checkpoint_path = Path(
    "/root/autodl-tmp/models/motip/"
    "r50_deformable_detr_motip_sportsmot.pth"
)

print("当前目录:", Path.cwd())
print("配置文件:", config_path)
print("配置存在:", config_path.is_file())
print("权重文件:", checkpoint_path)
print("权重存在:", checkpoint_path.is_file())

assert config_path.is_file(), config_path
assert checkpoint_path.is_file(), checkpoint_path

config = yaml_to_dict(str(config_path))

# SportsMOT 配置还会读取父配置，所以当前目录必须是 MOTIP_ROOT
config = load_super_config(
    config,
    config["SUPER_CONFIG_PATH"],
)

dtype = torch.float16

model, _ = build_model(config)
load_checkpoint(model, str(checkpoint_path))

model.eval()
model = model.cuda()

if dtype == torch.float16:
    model.half()

print("SportsMOT model built successfully.")

### Process the video

In [ ]:
from pathlib import Path

import cv2
import torch
from tqdm import tqdm
from torchvision.transforms import functional as F

from utils.nested_tensor import nested_tensor_from_tensor_list
from demo.colormap import get_color
from models.runtime_tracker import RuntimeTracker


input_dir = Path(
    "/root/autodl-tmp/data/basket_cut/pl_nba_smoke"
)

output_dir = Path(
    "/root/autodl-tmp/outputs/basket_cut/motip/pl_nba_smoke"
)

output_dir.mkdir(parents=True, exist_ok=True)

video_paths = sorted(input_dir.glob("*.mp4"))

print(f"Found {len(video_paths)} videos:")
for path in video_paths:
    print(" -", path.name)


def simple_transform(image, max_shorter, max_longer, image_dtype):
    image = F.to_tensor(image)
    image = F.resize(
        image,
        size=max_shorter,
        max_size=max_longer,
    )
    image = F.normalize(
        image,
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )

    if image_dtype != torch.float32:
        image = image.to(image_dtype)

    return image.cuda()


for video_path in video_paths:
    output_path = output_dir / (
        f"{video_path.stem}_motip_sportsmot.mp4"
    )

    print(f"\nProcessing: {video_path.name}")

    video_cap = cv2.VideoCapture(str(video_path))

    if not video_cap.isOpened():
        print(f"Failed to open: {video_path}")
        continue

    fps = video_cap.get(cv2.CAP_PROP_FPS)
    width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    length = int(video_cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps <= 0:
        fps = 25.0

    print(
        f"frames={length}, fps={fps:.2f}, "
        f"resolution={width}x{height}"
    )

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    video_writer = cv2.VideoWriter(
        str(output_path),
        fourcc,
        fps,
        (width, height),
    )

    # 每条视频都必须新建一个 Tracker，防止上一条视频的轨迹状态泄漏
    runtime_tracker = RuntimeTracker(
        model=model,
        sequence_hw=(height, width),
        assignment_protocol="object-max",
        miss_tolerance=60,
        det_thresh=0.3,
        newborn_thresh=0.6,
        id_thresh=0.2,
        dtype=dtype,
    )

    with torch.inference_mode():
        for _ in tqdm(
            range(length),
            desc=video_path.stem,
            unit="frame",
        ):
            ret, frame = video_cap.read()

            if not ret:
                break

            rgb_frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB,
            )

            frame_tensor = simple_transform(
                rgb_frame,
                max_shorter=800,
                max_longer=1440,
                image_dtype=dtype,
            )

            frame_tensor = nested_tensor_from_tensor_list(
                [frame_tensor]
            )

            runtime_tracker.update(frame_tensor)
            track_results = runtime_tracker.get_track_results()

            for bbox, obj_id in zip(
                track_results["bbox"],
                track_results["id"],
            ):
                x, y, w, h = map(int, bbox)

                color = get_color(
                    obj_id,
                    rgb=False,
                    use_int=True,
                )

                cv2.rectangle(
                    frame,
                    (x, y),
                    (x + w, y + h),
                    color,
                    2,
                )

                cv2.putText(
                    frame,
                    f"ID: {obj_id}",
                    (x, max(y - 10, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    color,
                    2,
                )

            video_writer.write(frame)

    video_cap.release()
    video_writer.release()

    del runtime_tracker
    torch.cuda.empty_cache()

    print(f"Saved: {output_path}")

print("\nAll videos completed.")